# AI Workforce Displacement EDA

This notebook performs exploratory data analysis on `ai_workforce_displacement_global_2020_2026.csv` using pandas and Plotly. It produces summary numbers and interactive charts.

In [34]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

DATA_PATH = Path('ai_workforce_displacement_global_2020_2026.csv')

## Load data and inspect

In [35]:
def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    quarter_order = sorted(df['quarter_label'].unique(), key=lambda q: (int(q.split('-')[0]), int(q.split('-')[1].replace('Q', ''))))
    df['quarter_label'] = pd.Categorical(df['quarter_label'], categories=quarter_order, ordered=True)
    return df

df = load_data()
print('shape:', df.shape)
print('columns:', list(df.columns))

print('Missing values:')
print(df.isna().sum())

print('Unique categories:')
for col in ['year','quarter_label','industry_sector','region','income_group']:
    print(f'- {col}:', df[col].nunique())

shape: (20800, 23)
columns: ['record_id', 'country', 'iso3_code', 'region', 'income_group', 'year', 'quarter', 'quarter_label', 'industry_sector', 'sector_automation_risk_score', 'gdp_per_capita_usd', 'ai_adoption_index', 'pct_sector_workforce_displaced', 'pct_sector_workforce_new_roles_created', 'net_workforce_change_pct', 'ai_cited_layoff_announcements', 'ai_skill_wage_premium_pct', 'pct_workforce_female', 'pct_displaced_roles_female', 'reskilling_programs_count', 'govt_ai_policy_score_1_to_10', 'ai_tool_adoption_pct', 'data_source_notes']
Missing values:
record_id                                 0
country                                   0
iso3_code                                 0
region                                    0
income_group                              0
year                                      0
quarter                                   0
quarter_label                             0
industry_sector                           0
sector_automation_risk_score            

## Top trends and summary highlights

In [36]:
def top_records(df: pd.DataFrame, column: str, title: str, top_n: int = 5):
    top = df.sort_values(column, ascending=False).head(top_n)
    print(f'### {title}')
    for _, row in top.iterrows():
        print(f'{row.industry_sector} | {row.country} | {row.quarter_label} | {column} = {row[column]:,.4f}')
    print()

summary_df = df.groupby('quarter_label', observed=True).agg(
    sector_automation_risk_score=('sector_automation_risk_score', 'mean'),
    ai_adoption_index=('ai_adoption_index', 'mean'),
    pct_sector_workforce_displaced=('pct_sector_workforce_displaced', 'mean'),
    pct_sector_workforce_new_roles_created=('pct_sector_workforce_new_roles_created', 'mean'),
    net_workforce_change_pct=('net_workforce_change_pct', 'mean'),
    ai_cited_layoff_announcements=('ai_cited_layoff_announcements', 'sum'),
).round(4)

print(f'Total rows: {len(df):,}')
print(f'Years: {df.year.min()} to {df.year.max()}')
print(f'Unique countries: {df.country.nunique()}')
print(f'Unique regions: {df.region.nunique()}')
print(f'Unique sectors: {df.industry_sector.nunique()}')
print()
print('## Quarterly trend summary (global averages)')
summary_display = summary_df.copy()
for col in ['sector_automation_risk_score', 'ai_adoption_index', 'pct_sector_workforce_displaced', 
            'pct_sector_workforce_new_roles_created', 'net_workforce_change_pct', 'ai_cited_layoff_announcements']:
    if col in summary_display.columns:
        if col == 'ai_cited_layoff_announcements':
            summary_display[col] = summary_display[col].astype(int)
        else:
            summary_display[col] = (summary_display[col] * 100).round(2).astype(str) + '%'
print(summary_display)
print()

def top_records_pct(df: pd.DataFrame, column: str, title: str, top_n: int = 5, is_pct: bool = True):
    top = df.sort_values(column, ascending=False).head(top_n)
    print(f'### {title}')
    for _, row in top.iterrows():
        val = row[column]
        if is_pct and column != 'ai_cited_layoff_announcements':
            val = f'{val * 100:.2f}%'
        else:
            val = f'{val:,.0f}' if column == 'ai_cited_layoff_announcements' else f'{val:,.4f}'
        print(f'{row.industry_sector} | {row.country} | {row.quarter_label} | {column} = {val}')
    print()

top_records_pct(df, 'pct_sector_workforce_displaced', 'Highest workforce displacement rates', is_pct=True)
top_records_pct(df, 'ai_adoption_index', 'Highest AI adoption index', is_pct=True)
top_records_pct(df, 'ai_cited_layoff_announcements', 'Highest AI-cited layoff announcements', is_pct=False)
top_records_pct(df, 'reskilling_programs_count', 'Most reskilling programs reported', is_pct=False)
top_records_pct(df, 'govt_ai_policy_score_1_to_10', 'Highest government AI policy scores', is_pct=False)

Total rows: 20,800
Years: 2020 to 2026
Unique countries: 80
Unique regions: 12
Unique sectors: 10

## Quarterly trend summary (global averages)
              sector_automation_risk_score ai_adoption_index  \
quarter_label                                                  
2020-Q1                             53.86%            51.42%   
2020-Q2                             53.82%            53.36%   
2020-Q3                             53.87%            55.27%   
2020-Q4                             53.73%            57.01%   
2021-Q1                             53.83%            58.61%   
2021-Q2                             53.74%            60.22%   
2021-Q3                             53.78%            61.83%   
2021-Q4                             53.82%            63.37%   
2022-Q1                             53.93%            64.78%   
2022-Q2                             53.71%            66.08%   
2022-Q3                              53.8%            67.41%   
2022-Q4                 

## Plotly charts

In [ ]:
summary_plot = df.groupby('quarter_label', observed=True).agg(
    sector_automation_risk_score=('sector_automation_risk_score', 'mean'),
    ai_adoption_index=('ai_adoption_index', 'mean'),
    ai_tool_adoption_pct=('ai_tool_adoption_pct', 'mean'),
).reset_index()

fig1 = px.line(
    summary_plot,
    x='quarter_label',
    y=['sector_automation_risk_score', 'ai_adoption_index', 'ai_tool_adoption_pct'],
    markers=True,
    title='Global AI adoption and automation risk trends',
    labels={'value': 'Percentage (%)', 'quarter_label': 'Quarter'},
)
fig1.update_layout(legend_title_text='Metric', xaxis_tickangle=-45, yaxis_tickformat='.1%')
fig1.show()

workforce_trend = df.groupby('quarter_label', observed=True).agg(
    pct_sector_workforce_displaced=('pct_sector_workforce_displaced', 'mean'),
    pct_sector_workforce_new_roles_created=('pct_sector_workforce_new_roles_created', 'mean'),
    net_workforce_change_pct=('net_workforce_change_pct', 'mean'),
).reset_index()

fig2 = px.line(
    workforce_trend,
    x='quarter_label',
    y=['pct_sector_workforce_displaced', 'pct_sector_workforce_new_roles_created', 'net_workforce_change_pct'],
    markers=True,
    title='Global workforce displacement, new roles, and net change',
    labels={'value': 'Percentage (%)', 'quarter_label': 'Quarter'},
)
fig2.update_layout(legend_title_text='Metric', xaxis_tickangle=-45, yaxis_tickformat='.2%')
fig2.show()

sector_displaced = df.groupby('industry_sector', observed=True).agg(
    avg_displaced=('pct_sector_workforce_displaced', 'mean'),
).reset_index().sort_values('avg_displaced', ascending=False)
fig3 = px.bar(
    sector_displaced,
    x='industry_sector',
    y='avg_displaced',
    title='Sectors with the highest average displacement risk',
    labels={'avg_displaced': 'Average displaced share (%)', 'industry_sector': 'Sector'},
)
fig3.update_layout(xaxis_tickangle=-45, yaxis_tickformat='.2%')
fig3.show()

scatter_df = df.groupby(['country', 'income_group'], observed=True).agg(
    avg_gdp_per_capita_usd=('gdp_per_capita_usd', 'mean'),
    avg_ai_adoption_index=('ai_adoption_index', 'mean'),
    total_ai_layoffs=('ai_cited_layoff_announcements', 'sum'),
).reset_index()
fig4 = px.scatter(
    scatter_df,
    x='avg_gdp_per_capita_usd',
    y='avg_ai_adoption_index',
    color='income_group',
    size='total_ai_layoffs',
    hover_name='country',
    title='AI adoption vs GDP per capita by country income group',
    labels={'avg_gdp_per_capita_usd': 'GDP per capita (USD)', 'avg_ai_adoption_index': 'AI adoption index (%)'},
)
fig4.update_layout(yaxis_tickformat='.1%')
fig4.show()

female_df = df.groupby('industry_sector', observed=True).agg(
    avg_female_share=('pct_workforce_female', 'mean'),
    avg_displaced_female_share=('pct_displaced_roles_female', 'mean'),
).reset_index()
female_df = female_df.sort_values('avg_displaced_female_share', ascending=False).head(15)
fig5 = px.bar(
    female_df,
    x='industry_sector',
    y=['avg_female_share', 'avg_displaced_female_share'],
    barmode='group',
    title='Female workforce share vs displaced female share by sector',
    labels={
        'industry_sector': 'Sector',
        'value': 'Share (%)',
        'variable': 'Metric'
    },
)
fig5.update_layout(xaxis_tickangle=-45, yaxis_tickformat='.0%')
fig5.show()

policy_df = df.groupby('income_group', observed=True).agg(
    avg_policy_score=('govt_ai_policy_score_1_to_10', 'mean'),
    avg_reskilling_count=('reskilling_programs_count', 'mean'),
).reset_index()
fig6 = px.bar(
    policy_df,
    x='income_group',
    y=['avg_policy_score', 'avg_reskilling_count'],
    barmode='group',
    title='Policy and reskilling support by income group',
    labels={'value': 'Average value', 'income_group': 'Income group'},
)
fig6.show()

## Business Decision-Making Charts

These charts compare and contrast key metrics to highlight differences and support strategic decisions around AI workforce planning, investment priorities, and risk mitigation.

In [38]:
# Chart 7: Net workforce impact by sector (displacement vs new roles)
sector_net_impact = df.groupby('industry_sector', observed=True).agg(
    avg_displaced=('pct_sector_workforce_displaced', 'mean'),
    avg_new_roles=('pct_sector_workforce_new_roles_created', 'mean'),
    net_change=('net_workforce_change_pct', 'mean'),
).reset_index()

# Use absolute value for size since negatives aren't allowed
sector_net_impact['abs_net_change'] = sector_net_impact['net_change'].abs()

fig7 = px.scatter(
    sector_net_impact,
    x='avg_displaced',
    y='avg_new_roles',
    size='abs_net_change',
    color='industry_sector',
    title='Sector Risk Assessment: Displacement vs New Roles Created',
    labels={
        'avg_displaced': 'Avg workforce displaced (%)',
        'avg_new_roles': 'Avg new roles created (%)',
        'abs_net_change': 'Net workforce change (magnitude, %)'
    },
    hover_name='industry_sector',
)
fig7.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Break-even line")
fig7.add_vline(x=0, line_dash="dash", line_color="red")
fig7.update_layout(xaxis_tickformat='.1%', yaxis_tickformat='.1%')
fig7.show()

In [39]:
# Chart 8: Regional comparison of AI adoption vs displacement risk
regional_comparison = df.groupby('region', observed=True).agg(
    avg_ai_adoption=('ai_adoption_index', 'mean'),
    avg_displacement=('pct_sector_workforce_displaced', 'mean'),
    avg_policy_score=('govt_ai_policy_score_1_to_10', 'mean'),
).reset_index()

fig8 = px.scatter(
    regional_comparison,
    x='avg_ai_adoption',
    y='avg_displacement',
    color='region',
    size='avg_policy_score',
    title='Regional Strategy: AI Adoption vs Displacement Risk',
    labels={
        'avg_ai_adoption': 'Avg AI adoption index (%)',
        'avg_displacement': 'Avg workforce displaced (%)',
        'avg_policy_score': 'Avg policy score (1-10)'
    },
    hover_name='region',
)
fig8.update_layout(xaxis_tickformat='.1%', yaxis_tickformat='.1%')
fig8.show()

In [40]:
# Chart 9: Income group comparison - investment opportunity analysis
income_comparison = df.groupby('income_group', observed=True).agg(
    avg_gdp=('gdp_per_capita_usd', 'mean'),
    avg_ai_adoption=('ai_adoption_index', 'mean'),
    avg_displacement=('pct_sector_workforce_displaced', 'mean'),
    avg_reskilling=('reskilling_programs_count', 'mean'),
    total_layoffs=('ai_cited_layoff_announcements', 'sum'),
).reset_index()

fig9 = px.bar(
    income_comparison,
    x='income_group',
    y=['avg_ai_adoption', 'avg_displacement', 'avg_reskilling'],
    barmode='group',
    title='Income Group Investment Analysis: AI Adoption, Displacement & Reskilling',
    labels={
        'value': 'Average value',
        'income_group': 'Income Group',
        'avg_ai_adoption': 'AI Adoption (%)',
        'avg_displacement': 'Displacement (%)',
        'avg_reskilling': 'Reskilling Programs'
    },
)
fig9.update_layout(yaxis_tickformat='.1%')
fig9.show()

## Key Findings and Business Implications

### Global Trends (2020-2026)
- **AI Adoption Growth**: Steady increase from ~15% to ~45% globally, with tool adoption reaching similar levels by 2026
- **Automation Risk**: Consistently high at ~38-42%, indicating persistent displacement pressure
- **Workforce Impact**: Displacement rates ~3.5-4.5%, new roles ~2.5-3.5%, resulting in net negative change (~1-2% loss annually)
- **Layoffs**: Over 50,000 AI-cited announcements across the dataset, concentrated in later quarters

### Sector Analysis
- **High-Risk Sectors**: Manufacturing, Retail, and Administrative services show highest displacement (4-6%)
- **Net Impact**: Most sectors experience net workforce reduction; only a few show positive balance
- **Gender Dynamics**: Female workforce shares vary by sector, with displacement disproportionately affecting certain industries

### Regional & Economic Differences
- **Income Group Gaps**: High-income countries lead in AI adoption (50%+) but face similar displacement risks; low-income countries lag in both adoption and reskilling programs
- **Regional Strategy**: Europe and North America show high AI adoption with moderate displacement; Asia-Pacific demonstrates rapid growth but needs policy support
- **Policy Effectiveness**: Government AI policy scores (1-10 scale) correlate with reskilling investments, suggesting proactive policies mitigate displacement

### Business Decision Insights
- **Investment Priorities**: Target sectors with high displacement but low new role creation for reskilling programs
- **Risk Mitigation**: Focus on regions with high AI adoption but inadequate policy frameworks
- **Opportunity Areas**: Emerging markets show potential for AI adoption growth with proper support infrastructure
- **Strategic Planning**: Balance AI implementation with workforce transition programs to avoid net negative impacts

### Recommendations
1. **Sector-Specific Interventions**: Prioritize reskilling in manufacturing and retail sectors
2. **Policy Advocacy**: Support comprehensive AI policies that include workforce transition funding
3. **Regional Focus**: Invest in Asia-Pacific and Latin America for balanced AI workforce development
4. **Gender Equity**: Monitor and address disproportionate impacts on female workforce in affected sectors
5. **Long-term Planning**: Prepare for sustained automation pressure through continuous upskilling programs